### Wikipedia Retriever

In [ ]:
!pip install langchain chromadb faiss-cpu openai tiktoken langchain_openai langchain-community wikipedia

In [ ]:
from langchain_community.retrievers import WikipediaRetriever

In [ ]:
# initialize the retreiever (optional: set language and top_k)
retriever = WikipediaRetriever(top_k_results=2, lang='en')

In [ ]:
# define your query
query = "the geopolitical history of India and Pakistan from the perspective of a chinese"

# Get relevant wikipedia documents
docs = retriever.invoke(query) # retriever is runnable as it has invoke property

In [ ]:
print(docs) # not proper to understand

In [ ]:
# Print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1} ---")
    print(f"Content: \n{doc.page_content}....") # truncate for display

### Vector Store Retriever

In [ ]:
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document

In [ ]:
# Step 1: Your Source documents
documents = [
    Document(page_content="lanhchain helps developers build LLM applications easily."),
    # generate three more regarding Chroma, embeddins 
]

In [ ]:
# step 2: Initialize embedding Model
embedding_model = OpenAIEmbeddings()

# step 3: Create Chroma Vector Store in memory
vectorstore = Chroma.from_documents(
    documents = documents,
    embedding = embedding_model,
    collection_name = "my_collection"
)

In [ ]:
# step 4: Convert vectorstore into a retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [ ]:
query = "What is Chroma used for?"
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)

In [ ]:
results = vectorstore.similarity_search(query,k=2) # this method can give only in one stratgy can't try out diffeent strategy, that's why above code needed, also above code has invoke priperty so can convert to chains, but major adv is the early point mentioned
for i, doc in enumerate(results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)

### MMR (Maximum Marginal Relevance)

In [ ]:
# sample documents
docs = [
    Document(page_content=""),
    # generate 5 more docs first and last one about Langchain, rest three would be abot chroma, embeddings, MMR
]

In [ ]:
from langchain_community.vectorstores import FAISS

# Initialize OpenAI embeddings
embedding_model = OpenAIEmbeddings()

# step 2: create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [ ]:
# enable MMR in the retriever
retriever = vectorstore.as_retriever(
    search_type = "mmr",         # <-- this enables MMR
    search_kwargs = {"k": 3, "lambda_mult": 1}   # k = top results, lambda_mult = relevance-diversity balance, fot lambda_mult = 0 very diverse result, be somewhat between 0 and 1, for lambda_mult=1, it will work as normal similarity search, it will not work for fetching relevant yet diversity
)

In [ ]:
query = "What is Langchain?"
results = retriever.invoke(query)

In [ ]:
for i, doc in enumerate(results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)

### Multi-Query Retriever

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_openai import ChatopenAI
from langchain.retrievers.multi_query import MultiQueryRetriever

In [ ]:
# relevant health and wellness documents
all_docs = [
    # 
]

In [ ]:
# initialize OpenAI Embeddings
embedding_model = OpenAIEmbeddings()

# step 2: create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [ ]:
# enable MMR in the retriever
similarity_retriever = vectorstore.as_retriever(
    search_type = "similarity",         # <-- this enables MMR
    search_kwargs = {"k": 5}   # k = top results, lambda_mult = relevance-diversity balance, fot lambda_mult = 0 very diverse result, be somewhat between 0 and 1, for lambda_mult=1, it will work as normal similarity search, it will not work for fetching relevant yet diversity
)

In [ ]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever = vectorstore.as_retriever(
    search_type = "similarity",         # <-- this enables MMR
    search_kwargs = {"k": 5}),
    llm = ChatOpenAI(model="gpt-3.5-turbo") # llm to split ambiguous query
)

In [ ]:
# retriever results
similarity_results = similarity_retriever.invoke(query)
multiquery_results = multiquery_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(similarity_results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)

In [ ]:
for i, doc in enumerate(multiquery_results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)

### Contextual Compression Retriever

In [ ]:
dos = [
    
]

In [ ]:
# initialize OpenAI Embeddings
embedding_model = OpenAIEmbeddings()

# step 2: create the FAISS vector store from documents
vectorstore = FAISS.from_documents(
    documents = docs,
    embedding = embedding_model
)

In [ ]:
# setup the Compressor using an LLM
llm = ChatOpenAI(model='gpt-3.5-turbo')
compressor = LLMChainExtractor.from_llm(llm)

In [ ]:
# create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever = base_retriever,
    base_compressor = compressor
)

In [ ]:
# query the retriever
query = "What is photosynthesis?"
compressed_results = compression_retriever.invoke(query)

In [ ]:
for i, doc in enumerate(compressed_results):
    print(f"\n--Result {i+1}---")
    print(doc.page_content)